In [ ]:
# Cài đặt spaCy (chỉ cần chạy một lần)
# Lưu ý: Trong Colab/Jupyter, bạn cần dùng ! trước lệnh terminal
!pip install -U spacy

# Tải về mô hình tiếng Anh (kích thước trung bình)
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 55.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
from spacy import displacy

# Tải mô hình tiếng Anh đã cài đặt
nlp = spacy.load("en_core_web_md")

# Câu ví dụ
text = "The quick brown fox jumps over the lazy dog."

# Phân tích câu với pipeline của spaCy
doc = nlp(text)

# Trực quan hóa cây phụ thuộc trực tiếp trong Notebook
print("Cây phụ thuộc:")
displacy.render(doc, style="dep", jupyter=True)
#

Cây phụ thuộc:


In [ ]:
# Lấy một câu khác để phân tích
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)

# In ra thông tin của từng token
print(f"{'TEXT':<12} | {'DEP':<10} | {'HEAD TEXT':<12} | {'HEAD POS':<8} | {'CHILDREN'}")
print("-" * 70)

for token in doc:
    # Trích xuất các thuộc tính
    children = [child.text for child in token.children]
    print(f"{token.text:<12} | {token.dep_:<10} | {token.head.text:<12} | {token.head.pos_:<8} | {children}")

TEXT         | DEP        | HEAD TEXT    | HEAD POS | CHILDREN
----------------------------------------------------------------------
Apple        | nsubj      | looking      | VERB     | []
is           | aux        | looking      | VERB     | []
looking      | ROOT       | looking      | VERB     | ['Apple', 'is', 'at']
at           | prep       | looking      | VERB     | ['buying']
buying       | pcomp      | at           | ADP      | ['startup']
U.K.         | compound   | startup      | NOUN     | []
startup      | dobj       | buying       | VERB     | ['U.K.', 'for']
for          | prep       | startup      | NOUN     | ['billion']
$            | quantmod   | billion      | NUM      | []
1            | compound   | billion      | NUM      | []
billion      | pobj       | for          | ADP      | ['$', '1']


In [ ]:
text = "The cat chased the mouse and the dog watched them."
doc = nlp(text)

print("--- Found Triplet (Subject, Verb, Object) ---")

for token in doc:
    # Chỉ tìm các động từ
    if token.pos_ == "VERB":
        verb = token.text
        subject = ""
        obj = ""

        # Tìm chủ ngữ (nsubj) và tân ngữ (dobj) trong các con của động từ
        for child in token.children:
            if child.dep_ == "nsubj":
                subject = child.text
            if child.dep_ == "dobj":
                obj = child.text

        if subject and obj:
            print(f"Found Triplet: ({subject}, {verb}, {obj})")

--- Found Triplet (Subject, Verb, Object) ---
Found Triplet: (cat, chased, mouse)
Found Triplet: (dog, watched, them)


In [ ]:
text = "The big, fluffy white cat is sleeping on the warm mat."
doc = nlp(text)

print("\n--- Finding Adjectives modifying Nouns ---")

for token in doc:
    # Chỉ tìm các danh từ
    if token.pos_ == "NOUN":
        adjectives = []

        # Tìm các tính từ bổ nghĩa (amod) trong các con của danh từ
        for child in token.children:
            if child.dep_ == "amod":
                adjectives.append(child.text)

        if adjectives:
            print(f"Danh từ '{token.text}' được bổ nghĩa bởi các tính từ: {adjectives}")


--- Finding Adjectives modifying Nouns ---
Danh từ 'cat' được bổ nghĩa bởi các tính từ: ['big', 'fluffy', 'white']
Danh từ 'mat' được bổ nghĩa bởi các tính từ: ['warm']


In [ ]:
def find_main_verb(doc):
    """Tìm và trả về Token là động từ chính (ROOT) của câu."""
    for token in doc:
        if token.dep_ == "ROOT":
            return token
    return None

# Ví dụ
text1 = "The quick brown fox jumps over the lazy dog."
doc1 = nlp(text1)
verb1 = find_main_verb(doc1)

print("--- Bài 1: Tìm Động từ Chính (ROOT) ---")
print(f"Câu: '{text1}' -> Động từ chính: {verb1.text if verb1 else 'Không tìm thấy'}")

--- Bài 1: Tìm Động từ Chính (ROOT) ---
Câu: 'The quick brown fox jumps over the lazy dog.' -> Động từ chính: jumps


In [ ]:
def get_simple_noun_chunks(doc):
    """Tự viết hàm trích xuất các cụm danh từ đơn giản."""
    noun_chunks = []

    for token in doc:
        # Bắt đầu từ một danh từ
        if token.pos_ == "NOUN":
            chunk_components = []

            # Lấy các từ bổ nghĩa cho danh từ đó (children)
            for child in token.children:
                # Kiểm tra các quan hệ bổ nghĩa thường thấy trong cụm danh từ
                if child.dep_ in ("det", "amod", "compound", "nummod", "poss"):
                    chunk_components.append(child)

            # Thêm danh từ chính
            chunk_components.append(token)

            # Sắp xếp các thành phần theo thứ tự xuất hiện trong câu
            chunk_components.sort(key=lambda t: t.i)

            # Ghép thành chuỗi
            chunk = " ".join([t.text for t in chunk_components])
            noun_chunks.append(chunk)

    return noun_chunks

# Ví dụ
text = "The quick brown fox jumps over the lazy dog."
doc = nlp(text)

print("\n--- Bài 2: Tự trích xuất Cụm danh từ ---")
print(f"Cụm danh từ (tự viết): {get_simple_noun_chunks(doc)}")
print(f"Kiểm tra với spaCy tích hợp: {[chunk.text for chunk in doc.noun_chunks]}")


--- Bài 2: Tự trích xuất Cụm danh từ ---
Cụm danh từ (tự viết): ['The quick brown fox', 'the lazy dog']
Kiểm tra với spaCy tích hợp: ['The quick brown fox', 'the lazy dog']


In [ ]:
def get_path_to_root(token):
    """Tìm đường đi từ một token bất kỳ lên đến gốc (ROOT)."""
    path = [token]
    # Dừng khi token hiện tại là ROOT
    while token.dep_ != "ROOT":
        token = token.head
        path.append(token)
    return path

# Ví dụ
text = "Apple is looking at buying U.K. startup."
doc = nlp(text)

# Lấy token 'startup'
startup_token = doc[6]

path = get_path_to_root(startup_token)

print("\n--- Bài 3: Tìm Đường đi đến ROOT ---")
print(f"Token Bắt đầu: '{startup_token.text}'")
print("Đường đi (từ dependent lên ROOT):")
# In ra đường đi theo định dạng Text (DEP)
path_texts = [f"{t.text} ({t.dep_})" for t in path]
print(" -> ".join(path_texts))


--- Bài 3: Tìm Đường đi đến ROOT ---
Token Bắt đầu: 'startup'
Đường đi (từ dependent lên ROOT):
startup (dobj) -> buying (pcomp) -> at (prep) -> looking (ROOT)
